# NewsBot Intelligence System 2.0 — 06 Conversational Interface

## Goal
Evaluate intent and slot handling, grounded response templates, and follow-up context.

This notebook imports reusable project modules rather than duplicating implementation logic.

## Intent and parameter evaluation
The compact authored set covers all required intents and reports sample size.

In [1]:
import pandas as pd
pd.read_csv('data/results/tables/conversation_eval.csv')

,query,expected_intent,predicted_intent,intent_correct,intent_confidence,slot_matches
0,show me tech news,search,search,1.0,0.75,"{""category"": true}"
1,summarize politics coverage,summarize,summarize,1.0,0.75,"{""category"": true}"
2,show negative technology stories,sentiment,sentiment,1.0,0.75,"{""category"": true, ""sentiment"": true}"
3,how have topic trends changed,topic_trend,topic_trend,1.0,0.75,{}
4,find articles about Apple,entity_lookup,entity_lookup,1.0,0.75,"{""entities"": true}"
5,compare tech and business,compare,compare,1.0,0.75,"{""comparison_targets"": true}"
6,find similar articles,similar_articles,similar_articles,1.0,0.75,{}
7,help,help,help,1.0,0.75,{}
8,show wellness news from this week,search,search,1.0,0.75,"{""category"": true, ""date_start"": true}"


## Initialize the integrated query system
Responses cite local records and never claim live-news access.

In [2]:
from src.system import NewsBot2IntegratedSystem
system = NewsBot2IntegratedSystem().fit()
system.query_interface('Show me tech news')

{'response': 'Found 300 matching historical articles with category=TECH.',
 'results': [{'article_id': 995,
   'title': 'Self-Driving Uber In Fatal Accident Had 6 Seconds To React Before Crash',
   'category': 'TECH',
   'date': '2018-05-24'},
  {'article_id': 1591,
   'title': 'Facebook Suspends 200 Apps Amid Data Misuse Investigation',
   'category': 'TECH',
   'date': '2018-05-14'},
  {'article_id': 1192,
   'title': 'Keyless Cars Have Killed More Than 2 Dozen People Since 2006: Report',
   'category': 'TECH',
   'date': '2018-05-14'},
  {'article_id': 1456,
   'title': "Boston Dynamics' 'Robot Dog' May Be Available For Sale Soon",
   'category': 'TECH',
   'date': '2018-05-12'},
  {'article_id': 1588,
   'title': "Professor Who Sold Facebook Data To Cambridge Analytica 'Sincerely Sorry'",
   'category': 'TECH',
   'date': '2018-04-23'}],
 'applied_filters': {'raw_query': 'Show me tech news', 'category': 'TECH'},
 'next_actions': ['Ask for a summary, sentiment, trend, comparison, or

## Multi-turn context
The second query inherits TECH because it does not name a new category.

In [3]:
first = system.query_interface('Show me tech news')
second = system.query_interface('What about negative ones?')
{'first_filters': first['applied_filters'], 'second_filters': second['applied_filters'], 'second_response': second['response']}

{'first_filters': {'raw_query': 'Show me tech news', 'category': 'TECH'},
 'second_filters': {'raw_query': 'What about negative ones?',
  'sentiment': 'negative',
  'topic_keyword': 'negative ones',
  'category': 'TECH',
  'inherited_category': True},
 'second_response': 'Found 0 matching historical articles with sentiment=negative, topic_keyword=negative ones, category=TECH.'}

## Intent-specific examples
Each response remains grounded in the stored corpus or explicitly states an unavailable capability.

In [4]:
queries = ['Summarize politics coverage', 'How have topic trends changed?', 'Compare tech and business', 'Find articles about Apple', 'Find similar articles', 'Help']
[{query: system.query_interface(query)['response']} for query in queries]

[{'Summarize politics coverage': "Summary of 300 matching historical articles with category=POLITICS: Can The Democrats Come Together In 2018? 2014's Voting Changes Are Reason for 2015 Reform Rather than playing partisan politics with the ballot box, state legislators and local elected officials should seek to improve the electoral process and access to the ballot in 2015. After all, politicians should not be choosing their voters; voters should be choosing their politicians."},
 {'How have topic trends changed?': 'Emerging topic IDs: topic_2, topic_3, topic_5; declining topic IDs: topic_1, topic_4, topic_0. These are historical corpus trends, not live news.'},
 {'Compare tech and business': "Local coverage comparison: {'TECH': {'articles': 300, 'latest_date': '2018-05-24'}, 'BUSINESS': {'articles': 300, 'latest_date': '2022-03-23'}}. Counts describe this balanced historical sample, not real-world importance."},
 {'Find articles about Apple': 'Found 3 matching historical articles with 

## Limitations and ethics
The source corpus is historical and predominantly English. Confidence is not truth; sentiment, topics, named entities, translation, summaries, and semantic similarity can be wrong. Entity co-occurrence is not a proven real-world relationship. Internal corpus corroboration is not independent fact-checking.